# Experimental test 3 Result

* vllm기준으로 accuracy와 f1 score를 테스트
    * fewshot 테스트에 활용할 질문의 개수  : 30
    * 소스코드 포함여부  : 'Y'           
    * 반복횟수 : 100회                
    * 시스템프롬프트 'sys_prompt10'
    * self-consistency 횟수 : 5
    * temperature : 0.01
    * 엑셀버전 : 'ver7'
* 이후 결과에 대해서 스코어 비교 진행 


In [2]:
import os
import pandas as pd
from config import config as conf
import re
import numpy as np
from sklearn import metrics



In [3]:
def sc_calc_acc_condition_with_temp_with_sc(llm_model, few_shot_n, test_n, q_src_yn, ver, p_ver, sc_num, temp, excel_ver):
    tmp = pd.DataFrame()
    df_eval = pd.DataFrame()
    acc_list = []
    path = f'{conf.DATA_PATH}/{conf.ANNO_RESULT}'
    file_list = os.listdir(path)
    opt_file = [x for x in file_list if x.startswith(f'sc_{llm_model}_result_{few_shot_n}_{test_n}_{q_src_yn}_{ver}_{p_ver}_{sc_num}_{temp}_{excel_ver}')]
    opt_file = [x for x in opt_file if x.endswith(f'.csv')]

    df = pd.DataFrame()

    
    if len(opt_file)>0 : 
        for f in opt_file:
            tmp = pd.read_csv(f'{path}/{f}', index_col =0)
            tmp = tmp.dropna()

            tmp['gold'] = tmp['answer'].apply(lambda x : re.sub(r'[^012]', '', x))
            tmp['o_result'] = tmp['result'].apply(lambda x : re.sub(r'[^012]', '', x))
            tmp = tmp[tmp['o_result'].isin(['1', '0', '2'])]

            
            gold_df = tmp[['id', 'gold']].drop_duplicates()
            chk_cnt = tmp.groupby(['id', 'o_result']).count().reset_index()[['id', 'o_result', 'question']]
            chk_cnt = chk_cnt.rename(columns = {'question': 'cnt'})
            chk_cnt = chk_cnt[chk_cnt['cnt'] == sc_num]
            chk_cnt = chk_cnt.sort_values(by = ['id', 'cnt'], ascending=[True, False]).groupby(['id']).head(1)
            df_eval = pd.merge(gold_df, chk_cnt, on = ['id'])

            print(f'size of the dataset : {df_eval.shape[0]}')
            df_eval['equal_yn'] = np.where(df_eval['gold']==df_eval['o_result'], 1, 0)
            acc = (df_eval['equal_yn'].sum()/df_eval.shape[0])*100  
            acc_list.append(acc)
            df = pd.concat([df, df_eval], axis =0)
            
        df['equal_yn'] = np.where(df['gold']==df['o_result'], 1, 0)
        y_true = df['o_result']
        y_pred = df['gold']
        print(metrics.classification_report(y_true, y_pred, digits=3))

        
        acc = (df['equal_yn'].sum()/df.shape[0])*100            
        print(f'{llm_model}_result_{few_shot_n}_{test_n}_{q_src_yn} : ', acc)
        return acc_list


In [4]:
    # process3 = Process(target=task, args=('v',              # llm_model
    #                                       4,                # few_shot_n
    #                                       30,                # test_n(# of question for test)
    #                                       'Y',              # q_src_yn 
    #                                       100,                # iteration num
    #                                       'sys_prompt10',   # prompt ver
    #                                       5,                # self-consistency number
    #                                       0.01,             # temperature
    #                                       'ver7'            # excel_verion
    #                                       ))

In [5]:
list_ =         sc_calc_acc_condition_with_temp_with_sc('v', 3, 40, 'Y', 100, 'sys_prompt10', 5,  0.01, 'ver7')
print(list_)

size of the dataset : 19
size of the dataset : 22
size of the dataset : 21
size of the dataset : 23
size of the dataset : 17
size of the dataset : 19
size of the dataset : 18
size of the dataset : 27
size of the dataset : 22
size of the dataset : 20
size of the dataset : 17
size of the dataset : 22
size of the dataset : 15
size of the dataset : 17
size of the dataset : 29
size of the dataset : 19
size of the dataset : 22
size of the dataset : 20
size of the dataset : 18
size of the dataset : 21
size of the dataset : 23
size of the dataset : 17
size of the dataset : 16
size of the dataset : 21
size of the dataset : 14
size of the dataset : 20
size of the dataset : 24
size of the dataset : 21
size of the dataset : 29
size of the dataset : 22
size of the dataset : 16
size of the dataset : 18
size of the dataset : 25
size of the dataset : 19
size of the dataset : 24
size of the dataset : 21
size of the dataset : 20
size of the dataset : 23
size of the dataset : 19
size of the dataset : 23
